# Simulação de Opinião Pública com LLMs

**Disciplina:** Inteligência Artificial, Mackenzie 2026/1  
**Professor:** Rogerio Oliveira  
**Grupo:** Matheus Henrique de Oliveira Santos e Marina Cantarelli Barroca

---

## Objetivo

Este trabalho utiliza um modelo de linguagem (LLM) para simular respostas de uma pesquisa de opinião pública, comparando as distribuições geradas pelo modelo com os dados reais dos respondentes.

A abordagem é baseada no artigo *Simulating Public Opinion: Comparing Distributional and Individual-Level Predictions from LLMs and Random Forests* (Argyle et al.), que propõe o uso de LLMs como *silicon sampling*: modelos que reproduzem padrões de resposta humana a partir do perfil sociodemográfico dos respondentes.

## 1. Instalação e Configuração

In [ ]:
!pip install -q google-generativeai pandas scikit-learn matplotlib seaborn

In [ ]:
import google.generativeai as genai
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix
import time
import io

# API Key gratuita em https://aistudio.google.com
# Salve como secret no Colab com o nome GEMINI_KEY
from google.colab import userdata
GEMINI_KEY = userdata.get('GEMINI_KEY')
genai.configure(api_key=GEMINI_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')

print('Configuração OK')

## 2. Dados da Pesquisa

Utilizamos a pesquisa **04839 - Percepção dos Brasileiros sobre Temas Relacionados à Desigualdade** do Cesop/Unicamp, coletada em julho de 2023 com 2.000 respondentes em todo o Brasil.

As questões simuladas foram:
- **Q1 (P6B):** *"A maior presença de pessoas negras e indígenas nas universidades, por meio de cotas, é necessária para reduzir a desigualdade no Brasil"* — Concorda / Neutro / Discorda
- **Q2 (P6E):** *"As mudanças climáticas e eventos extremos afetam mais as pessoas em situação de pobreza"* — Concorda / Neutro / Discorda

Variáveis de perfil: sexo, escolaridade, renda, região e idade.

In [ ]:
# Carrega os dados reais da pesquisa Cesop 04839
# Fonte: https://www.cesop.unicamp.br
# IMPORTANTE: faça upload do arquivo 'dados_cesop_04839.csv' no Colab antes de rodar

import os
if not os.path.exists('dados_cesop_04839.csv'):
    from google.colab import files
    print('Faça upload do arquivo dados_cesop_04839.csv:')
    files.upload()

df = pd.read_csv('dados_cesop_04839.csv')
print(f'Dataset: {len(df)} respondentes')
df.head()

In [ ]:
# Distribuição real das respostas
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Distribuição Real - Pesquisa Cesop 04839', fontsize=13)

df['q1_cotas'].value_counts(normalize=True).plot(
    kind='bar', ax=axes[0], color='steelblue',
    title='Q1: Cotas nas Universidades'
)
df['q2_clima'].value_counts(normalize=True).plot(
    kind='bar', ax=axes[1], color='steelblue',
    title='Q2: Mudanças Climáticas e Pobreza'
)
for ax in axes:
    ax.set_ylabel('Proporção')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 3. Simulação com LLM

Para cada respondente, construímos um prompt com o perfil sociodemográfico e pedimos ao Gemini que simule as respostas. O modelo é solicitado a responder apenas com as opções disponíveis, sem texto adicional.

In [ ]:
def construir_prompt(row):
    idade = int(row['idade']) if not pd.isna(row['idade']) else 'não informada'
    return f"""Você está simulando a resposta de uma pessoa brasileira com o seguinte perfil:
- Sexo: {row['sexo']}
- Idade: {idade} anos
- Escolaridade: {row['escolaridade']}
- Renda familiar: {row['renda']}
- Região: {row['regiao']}

Responda as perguntas como essa pessoa responderia, considerando seu contexto social e econômico.
Responda APENAS no formato abaixo, sem explicações:
Q1: [resposta]
Q2: [resposta]

Pergunta 1: A maior presença de pessoas negras e indígenas nas universidades, por meio de cotas, é necessária para reduzir a desigualdade no Brasil.
Opções: Concorda / Neutro / Discorda

Pergunta 2: As mudanças climáticas e eventos extremos, como chuvas intensas e secas, afetam mais as pessoas em situação de pobreza.
Opções: Concorda / Neutro / Discorda"""


def extrair_respostas(texto):
    q1, q2 = None, None
    opcoes_validas = {'Concorda', 'Neutro', 'Discorda'}
    for linha in texto.strip().split('\n'):
        if linha.startswith('Q1:'):
            val = linha.replace('Q1:', '').strip()
            q1 = val if val in opcoes_validas else None
        elif linha.startswith('Q2:'):
            val = linha.replace('Q2:', '').strip()
            q2 = val if val in opcoes_validas else None
    return q1, q2


def simular_rodada(df_full, seed=42, n=200):
    sub = df_full.sample(n, random_state=seed).reset_index(drop=True)
    respostas_q1, respostas_q2 = [], []

    for i, row in sub.iterrows():
        try:
            prompt = construir_prompt(row)
            resposta = model.generate_content(prompt)
            q1, q2 = extrair_respostas(resposta.text)
        except Exception as e:
            q1, q2 = None, None
        respostas_q1.append(q1)
        respostas_q2.append(q2)
        time.sleep(0.5)  # respeita o rate limit do free tier

    sub['sim_q1'] = respostas_q1
    sub['sim_q2'] = respostas_q2
    return sub

print('Funções definidas.')

In [ ]:
# 3 repetições com sementes diferentes
seeds = [42, 7, 13]
resultados = []

for s in seeds:
    print(f'Rodada seed={s}...')
    res = simular_rodada(df, seed=s, n=200)
    validas = res['sim_q1'].notna().sum()
    resultados.append(res)
    print(f'  Concluída: {validas}/200 respostas válidas')

print('\nSimulação concluída.')

## 4. Análise e Comparação dos Resultados

In [ ]:
# Distribuição real vs simulada (primeira rodada)
res_rodada1 = resultados[0].dropna(subset=['sim_q1', 'sim_q2'])
ordem = ['Concorda', 'Neutro', 'Discorda']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Distribuição Real vs Simulada', fontsize=14)

res_rodada1['q1_cotas'].value_counts(normalize=True).reindex(ordem).plot(
    kind='bar', ax=axes[0,0], color='steelblue', title='Q1 - Real'
)
res_rodada1['sim_q1'].value_counts(normalize=True).reindex(ordem).plot(
    kind='bar', ax=axes[0,1], color='coral', title='Q1 - Simulado'
)
res_rodada1['q2_clima'].value_counts(normalize=True).reindex(ordem).plot(
    kind='bar', ax=axes[1,0], color='steelblue', title='Q2 - Real'
)
res_rodada1['sim_q2'].value_counts(normalize=True).reindex(ordem).plot(
    kind='bar', ax=axes[1,1], color='coral', title='Q2 - Simulado'
)

for ax in axes.flat:
    ax.set_ylabel('Proporção')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Acurácia por rodada
acuracias_q1, acuracias_q2 = [], []

print('Acurácia por rodada:\n')
for i, rodada in enumerate(resultados):
    r = rodada.dropna(subset=['sim_q1', 'sim_q2'])
    acc_q1 = accuracy_score(r['q1_cotas'], r['sim_q1'])
    acc_q2 = accuracy_score(r['q2_clima'], r['sim_q2'])
    acuracias_q1.append(acc_q1)
    acuracias_q2.append(acc_q2)
    print(f'  Rodada {i+1} (seed={seeds[i]}): Q1={acc_q1:.2%} | Q2={acc_q2:.2%}')

print(f'\nMédia Q1: {np.mean(acuracias_q1):.2%} (±{np.std(acuracias_q1):.2%})')
print(f'Média Q2: {np.mean(acuracias_q2):.2%} (±{np.std(acuracias_q2):.2%})')

In [ ]:
# Matriz de confusão Q1
rodada1_cm = resultados[0].dropna(subset=['sim_q1'])
ordem = ['Concorda', 'Neutro', 'Discorda']
cm = confusion_matrix(rodada1_cm['q1_cotas'], rodada1_cm['sim_q1'], labels=ordem)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=ordem, yticklabels=ordem, cmap='Blues')
plt.title('Matriz de Confusão - Q1 (Cotas)')
plt.ylabel('Real')
plt.xlabel('Simulado')
plt.tight_layout()
plt.show()

In [ ]:
# Acurácia Q1 por escolaridade
rodada1_esc = resultados[0].dropna(subset=['sim_q1'])
grupos = rodada1_esc.groupby('escolaridade').apply(
    lambda g: accuracy_score(g['q1_cotas'], g['sim_q1'])
).reset_index()
grupos.columns = ['Escolaridade', 'Acurácia']
grupos = grupos.sort_values('Acurácia', ascending=False)

plt.figure(figsize=(8, 4))
plt.bar(grupos['Escolaridade'], grupos['Acurácia'], color='steelblue')
plt.title('Acurácia Q1 por Escolaridade')
plt.ylabel('Acurácia')
plt.ylim(0, 1)
plt.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()
print(grupos.to_string(index=False))

In [ ]:
# Acurácia Q1 por região
rodada1_reg = resultados[0].dropna(subset=['sim_q1'])
por_regiao = rodada1_reg.groupby('regiao').apply(
    lambda g: accuracy_score(g['q1_cotas'], g['sim_q1'])
).reset_index()
por_regiao.columns = ['Região', 'Acurácia']
por_regiao = por_regiao.sort_values('Acurácia', ascending=False)

plt.figure(figsize=(8, 4))
plt.bar(por_regiao['Região'], por_regiao['Acurácia'], color='teal')
plt.title('Acurácia Q1 por Região')
plt.ylabel('Acurácia')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()
print(por_regiao.to_string(index=False))

## 5. Discussão dos Resultados

**[Preencher após rodar o notebook com os resultados reais]**

Pontos a discutir:
- A distribuição simulada se aproximou da distribuição real? Em qual questão o modelo foi mais preciso?
- Algum grupo (escolaridade, região) teve acurácia significativamente diferente? O que pode explicar?
- Qual foi a variância entre as 3 repetições? O modelo é estável?
- Limitações: o LLM pode ter vieses de treinamento que distorcem a simulação de grupos menos representados.

## 6. Conclusão

**[Preencher após análise dos resultados]**

## Referências

ARGYLE, L. P. et al. *Simulating Public Opinion: Comparing Distributional and Individual-Level Predictions from LLMs and Random Forests*. Disponível na pasta do professor.

CESOP/UNICAMP. *04839 - Percepção dos Brasileiros sobre Temas Relacionados à Desigualdade*, 2023. Disponível em: https://www.cesop.unicamp.br. Acesso em: mai. 2026.

GOOGLE. *Gemini API Documentation*. Disponível em: https://ai.google.dev. Acesso em: mai. 2026.